In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Task 3 and 4_Loan_Data (1).csv")

df = df.sort_values("fico_score").reset_index(drop=True)

fico = df["fico_score"].values
default = df["default"].values

n = len(df)

In [2]:
cum_defaults = np.cumsum(default)
cum_total = np.arange(1, n + 1)

In [3]:
def bucket_stats(i, j):
    """
    Returns:
        observations
        defaults
    for interval [i,j]
    """

    total = j - i + 1

    defaults = cum_defaults[j]

    if i > 0:
        defaults -= cum_defaults[i - 1]

    return total, defaults

In [4]:
def bucket_loglikelihood(i, j):

    n_bucket, k_bucket = bucket_stats(i, j)

    if k_bucket == 0 or k_bucket == n_bucket:
        return 0

    p = k_bucket / n_bucket

    return (
        k_bucket * np.log(p)
        + (n_bucket - k_bucket) * np.log(1 - p)
    )

In [5]:
num_buckets = 10

dp = np.full((num_buckets + 1, n + 1), -np.inf)

split = np.zeros((num_buckets + 1, n + 1), dtype=int)

dp[0][0] = 0

In [6]:
for b in range(1, num_buckets + 1):

    for i in range(1, n + 1):

        for j in range(b - 1, i):

            score = (
                dp[b - 1][j]
                + bucket_loglikelihood(j, i - 1)
            )

            if score > dp[b][i]:
                dp[b][i] = score
                split[b][i] = j

In [7]:
boundaries = []

b = num_buckets
i = n

while b > 0:

    j = split[b][i]

    boundaries.append(j)

    i = j
    b -= 1

boundaries.reverse()

In [8]:
fico_boundaries = []

for idx in boundaries[1:]:

    fico_boundaries.append(
        fico[idx]
    )

print(fico_boundaries)

[np.int64(502), np.int64(553), np.int64(580), np.int64(608), np.int64(608), np.int64(648), np.int64(649), np.int64(696), np.int64(732)]


In [9]:
def fico_to_rating(score, boundaries):

    for i, boundary in enumerate(boundaries):

        if score < boundary:
            return len(boundaries) - i + 1

    return 1

In [10]:
fico_to_rating(810, fico_boundaries)

1

In [11]:
fico_to_rating(550, fico_boundaries)

9

In [12]:
df["rating"] = pd.qcut(
    df["fico_score"],
    q=10,
    labels=False
) + 1

In [13]:
df["rating"] = 11 - df["rating"]